# 03b - Rekayasa Fitur & Transformasi Variabel
Materi Modul 3: transformasi variabel, pengkodean fitur, penskalaan fitur.
Dataset: **Pima Indians Diabetes** (`dataset/diabetes.csv`, 768 sampel, 8 fitur).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import PowerTransformer, OrdinalEncoder, LabelEncoder, StandardScaler, MinMaxScaler, RobustScaler, PolynomialFeatures

df = pd.read_csv('../dataset/diabetes.csv')

## 1. Transformasi variabel (mengatasi skewness)

In [2]:
kolom = 'diabetes_pedigree'
x = df[kolom]
cand = {
    'Asli': x,
    'log1p': np.log1p(x),
    'sqrt': np.sqrt(x),
    'Yeo-Johnson': pd.Series(PowerTransformer().fit_transform(x.to_frame()).ravel()),
}
fig, ax = plt.subplots(1, 4, figsize=(16,3.5))
for a, (n, v) in zip(ax, cand.items()):
    a.hist(v, bins=20); a.set_title(f'{n} (skew={v.skew():.2f})')
plt.tight_layout(); plt.show()

## 2. Pengkodean fitur kategorikal
Diabetes hanya punya fitur numerik, jadi kita buat fitur kategorikal dari `age` lewat binning.

In [3]:
df['kelompok_usia'] = pd.cut(df['age'], bins=[0, 30, 45, 120], labels=['muda','dewasa','paruh_baya'])
df[['age','kelompok_usia']].head()

,age,kelompok_usia
0,50,paruh_baya
1,31,dewasa
2,32,dewasa
3,21,muda
4,33,dewasa


In [4]:
# One-hot encoding (tanpa urutan)
pd.get_dummies(df['kelompok_usia'], prefix='usia').head()

,usia_muda,usia_dewasa,usia_paruh_baya
0,False,False,True
1,False,True,False
2,False,True,False
3,True,False,False
4,False,True,False


In [5]:
# Ordinal encoding (ada urutan muda < dewasa < paruh_baya)
oe = OrdinalEncoder(categories=[['muda','dewasa','paruh_baya']])
df['usia_ordinal'] = oe.fit_transform(df[['kelompok_usia']]).astype(int)
# Label encoding untuk target berupa teks
le = LabelEncoder()
print(le.fit_transform(['a','b','c','a']), le.classes_)
df[['kelompok_usia','usia_ordinal']].drop_duplicates().sort_values('usia_ordinal')

[0 1 2 0] ['a' 'b' 'c']


,kelompok_usia,usia_ordinal
3,muda,0
1,dewasa,1
0,paruh_baya,2


## 3. Penskalaan fitur

In [6]:
fit = ['glucose','insulin']
skalers = {'Asli': None, 'Standard': StandardScaler(), 'MinMax': MinMaxScaler(), 'Robust': RobustScaler()}
fig, ax = plt.subplots(1, 4, figsize=(16,3.5))
for a, (n, s) in zip(ax, skalers.items()):
    d = df[fit] if s is None else pd.DataFrame(s.fit_transform(df[fit]), columns=fit)
    d.boxplot(ax=a); a.set_title(n)
plt.tight_layout(); plt.show()

## 4. Fitur interaksi / polinomial

In [7]:
pf = PolynomialFeatures(degree=2, include_bias=False)
P = pf.fit_transform(df[['glucose','bmi']])
pd.DataFrame(P, columns=pf.get_feature_names_out()).head()

,glucose,bmi,glucose^2,glucose bmi,bmi^2
0,148.0,33.6,21904.0,4972.8,1128.96
1,85.0,26.6,7225.0,2261.0,707.56
2,183.0,23.3,33489.0,4263.9,542.89
3,89.0,28.1,7921.0,2500.9,789.61
4,137.0,43.1,18769.0,5904.7,1857.61


## Catatan
- (kapan pakai log vs sqrt, one-hot vs ordinal, Standard vs MinMax vs Robust)